Cell 1 -- Create/Open the experiment project

In [ ]:
from pathlib import Path
from awr2944_dca import RadarProject, RadarProfile

ROOT = Path(r"C:\Users\khams008\Documents\saeed_room_measurement")

if (ROOT / "awr2944.toml").exists():
    p = RadarProject.open(ROOT)
else:
    p = RadarProject.create_at(ROOT)

print("Experiment project:", ROOT)

Cell 2 -- Connect/preflight

In [ ]:
# Detect the already-connected hardware/toolchain and save locally
serial = p.hardware.autodetect_serial(save=True)
tools  = p.hardware.autodetect_toolchain(save=True)

print(serial)
print(tools)

print("\nEthernet:")
print(p.eth.status())

print("\nDCA:")
print(p.dca.verify())

print("\nFull doctor:")
report = p.doctor()
report.print()

Cell 3 -- Define our actual experiment profile

In [ ]:
room_profile = (
    RadarProfile.smoke_v1()
    .with_frame(
        frame_count=32,
        chirps_per_frame=128,
    )
    .rename("room_static_v1")
)

p.profiles.save(room_profile)

print(room_profile.summary())
print("\nValidation:", room_profile.validate())

In [ ]:
plan = p.capture.plan(
    profile="room_static_v1",
    frames=32,
    guard_frames=1,
)

print(plan)
print("\nCube shape:", plan.cube_shape)
print("Hardware touched:", plan.hardware_touched)

Cell 4 -- ONE clean capture

In [ ]:
result = p.capture.run(
    profile="room_static_v1",
    name="room_static_01",
    frames=32,
    guard_frames=1,
)

print("Success:", result.success)
print("Capture:", result.capture.capture_id)
print("Path:", result.capture.path)

cap = result.capture

In [ ]:
verification = cap.verify()
print(verification)

assert verification.success

Cell 5 -- DSP + plots

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt

from awr2944_dca.dsp import run_pipeline
from awr2944_dca.dsp.visualization import (
    plot_raw_adc_traces,
    plot_range_profiles,
    plot_range_time_heatmap,
    plot_range_doppler,
    plot_range_doppler_average,
    plot_frame_chirp_rx_rms,
)

cube = cap.raw.to_cube("canonical")
dsp_profile = room_profile.to_dsp_profile()

result_dsp = run_pipeline(cube)

rng_db = result_dsp.range_result.power_db
rd_db  = result_dsp.doppler_result.rx_combined_power

out = cap.path / "analysis"
out.mkdir(exist_ok=True)

plots = [
    plot_raw_adc_traces(
        cube, dsp_profile,
        frame=0, chirp=0,
        save_path=out / "raw_adc"
    ),

    plot_range_profiles(
        rng_db, dsp_profile,
        frame=0,
        save_path=out / "range_profiles"
    ),

    plot_range_time_heatmap(
        rng_db, dsp_profile,
        rx=0,
        save_path=out / "range_time"
    ),

    plot_range_doppler(
        rd_db, dsp_profile,
        frame=0,
        save_path=out / "range_doppler"
    ),

    plot_range_doppler_average(
        rd_db, dsp_profile,
        save_path=out / "range_doppler_average"
    ),

    plot_frame_chirp_rx_rms(
        cube, dsp_profile,
        save_path=out / "rms"
    ),
]

for fig in plots:
    display(fig)
    plt.close(fig)

print("Saved plots:", out)

Cell 6 -- GUI

In [ ]:
cap.open_viewer()